# Stage 2 · LoRA 파인튜닝 → 검증 → 제출

**핵심 설계**
- **정답 토큰만 학습**: 추론 때와 똑같은 프롬프트를 넣고, 마지막 위치(답이 나올 자리)의 로짓에만 교차 엔트로피를 겁니다. 베이스라인처럼 질문·이미지 토큰까지 손실에 넣지 않으므로 기울기가 전부 "어느 글자를 고를까"에 쓰입니다. 학습과 추론의 입력이 완전히 같다는 것도 장점입니다.
- **선지 셔플 증강**: 매 epoch마다 선지 순서를 무작위로 섞고 정답 글자를 그에 맞춰 바꿉니다 → 자리 편향 제거, 같은 문항도 매번 다른 모양.
- **끊겨도 이어서**: `SAVE_EVERY` 스텝마다 어댑터·옵티마이저·진행 상태를 Drive에 저장합니다. 같은 셀을 다시 실행하면 이어서 학습합니다.
- 검증 분할·프롬프트·채점 방식은 Stage 1과 동일합니다 (점수 직접 비교 가능).

**실행 순서**: ① `DRY_RUN = True`로 위에서부터 끝까지 (3~5분, 오류·메모리·예상 시간 확인) → ② `DRY_RUN = False`로 바꿔 1번 셀부터 다시 실행 (본 학습).

## 1. 실험 설정

In [ ]:
RUN_NAME   = "q35_27b_r16_native_bf16"        # 실험 이름 = 저장 폴더 이름. 설정을 바꾸면 이름도 바꾸세요
MODEL_ID   = "Qwen/Qwen3.5-27B"
DRY_RUN    = False          # True: 3스텝만 학습해 전체 흐름 점검 (본 학습 전 필수) / False: 본 학습

# ── 학습 ──
TRAIN_PRESET = "native"    # 학습 해상도
EPOCHS     = 1
MAX_STEPS  = None          # 숫자를 넣으면 그 스텝에서 학습을 끝냄 (큰 모델을 짧게 학습할 때. 예: 200). None이면 EPOCHS만큼
LR         = 1e-4
MICRO_BS   = 2             # GPU에 한 번에 올리는 문항 수. 학습 중 OOM이 나면 자동으로 절반으로 줄이고 GRAD_ACCUM을 2배로 늘립니다
GRAD_ACCUM = 8             # 유효 배치 = MICRO_BS x GRAD_ACCUM = 16 (이 곱이 같으면 학습 결과의 성격은 같습니다)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
WARMUP_RATIO = 0.05
SHUFFLE_OPTIONS = True     # 선지 셔플 증강
TRAIN_ON_FULL = False      # True: 검증셋까지 합쳐 전체 6,714개로 학습 (레시피가 확정된 뒤 최종 제출용)
LOG_EVERY, EVAL_EVERY, SAVE_EVERY = 10, 100, 50    # 옵티마이저 스텝 단위
QUICK_EVAL_N = 300         # 학습 도중 빠른 점검용 검증 문항 수 (epoch 끝에는 1,000개 전체 평가)

# ── 평가/제출 ──
EVAL_PRESETS = ["up2.0"]   # 학습이 끝난 뒤 이 해상도들로 검증셋 전체 평가
EVAL_TTA     = False        # 가장 좋았던 해상도에서 선지 순열 4종 평균(TTA)의 효과도 측정
MAKE_SUBMISSION = False     # 학습·평가가 끝나면 이어서 test 추론 → 제출 파일까지 생성
TEST_PRESET  = "up2.0"      # "auto": 검증 정확도가 가장 높았던 해상도 / 또는 "native", "up1.5" 직접 지정
TEST_TTA     = False       # True면 test 추론 시간이 4배. 검증에서 TTA 이득이 확인된 뒤 켜서 12번 셀만 다시 실행

DRIVE_ZIP    = "/content/drive/MyDrive/ssafy-16-2-ai.zip"
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/content"
OUT_DIR      = "/content/drive/MyDrive/vqa_runs"
USE_4BIT   = False         # 4bit 없이 bf16 LoRA — H100(80GB)·G4(96GB) 전용. A100 40GB에서는 메모리 부족
BATCH_SIZE = "auto"        # 평가용 배치
SEED, VAL_SIZE = 42, 1000  # 검증 분할용 — 팀 공통값이므로 절대 바꾸지 마세요
TRAIN_SEED = 42            # 학습 순서·선지 셔플·LoRA 초기화 시드. 같은 레시피로 두 번째 모델을 만들 때(시드 앙상블) 43 등으로 변경
TRAIN_LIMIT = None         # 디버그용: 학습 샘플 수 제한
EXTRA_TRAIN_CSV = None     # 추가 학습 데이터 CSV 경로 (예: Stage 5가 만든 dev 의사 라벨). 열: id,path,question,a,b,c,d,answer. 검증 문항은 자동 차단
IMG_AUG = False            # True: 학습 이미지에만 밝기·대비·채도·감마·JPEG 화질을 무작위로 흔듦 (글자·방향·위치는 그대로). 평가·test에는 적용 안 함
TRAIN_PRESETS = None       # 예: ["native", "up1.5", "up2.0"] → 마이크로 배치마다 학습 해상도를 무작위로 고름(다해상도 학습). None이면 TRAIN_PRESET 고정
SKIP_MISSING_IMAGES = False
HF_TOKEN   = ""            # Gemma처럼 라이선스 승인이 필요한 모델일 때 HF 토큰 (비워 두면 Colab 보안 비밀 'HF_TOKEN'을 찾아봄)

RES_PRESETS = {
    "baseline384": (384 * 384, 384 * 384),
    "mid":         (256 * 28 * 28, 512 * 28 * 28),
    "native":      (256 * 28 * 28, 1280 * 28 * 28),
    "up1.5":       (1_550_000, 2_100_000),
    "up2.0":       (2_700_000, 3_400_000),
}
# Gemma 4 계열은 이미지당 '소프트 토큰' 수(70/140/280/560/1120)로 해상도를 정합니다. 프리셋 이름은 그대로 쓰고 값만 대응 (280 ≈ 원본, 560 ≈ 1.3배, 1120 ≈ 1.9배)
RES_PRESETS_GEMMA4 = {"baseline384": 70, "mid": 140, "native": 280, "up1.5": 560, "up2.0": 1120}
RES_PRESET = TRAIN_PRESET
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # GPU 메모리 단편화 완화 (torch import 전에 설정)

## 2. 라이브러리 설치 (버전 고정)

In [ ]:
import re
!pip -q install "transformers==5.17.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "peft==0.21.0"
# Colab에 기본 설치된 구버전 torchao가 있으면 peft 0.21이 LoRA를 붙이는 순간 ImportError를 냅니다. 이 프로젝트에서는 쓰지 않으므로 제거
!pip -q uninstall -y torchao
# Qwen3.5의 선형 어텐션용 고속 Triton 커널. 없으면 순수 PyTorch 대체 구현으로 돌아 학습 시 메모리 부족(OOM)이 나고 매우 느립니다.
# (transformers가 import 시점에 이 패키지를 찾으므로, 이미 모델을 로드한 세션에서 설치했다면 '세션 다시 시작'이 필요합니다)
if re.search(r"Qwen3\.[5-9]", MODEL_ID):      # Qwen3.5 / 3.6 / 3.8 — 모두 같은 선형 어텐션 구조(model_type qwen3_5)
    !pip -q install "flash-linear-attention==0.5.2"

## 3. 데이터 준비

In [ ]:
import os, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    print("압축 해제 중... (몇 분 걸립니다)")
    r = subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", DRIVE_ZIP],
                       capture_output=True, text=True)
    print(r.stdout[-400:]); assert r.returncode == 0, r.stderr[-800:]
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN, add_to_git_credential=False); print("Hugging Face 로그인 완료")
os.makedirs(OUT_DIR, exist_ok=True)
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} 가 {DATA_DIR} 에 없습니다. DATA_DIR / 압축 비밀번호를 확인하세요."
print("데이터 OK:", DATA_DIR, "| 결과 저장:", OUT_DIR)

## 4. 공통 import · CSV 로드

In [ ]:
import re, json, time, math, hashlib, random, datetime, platform
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def _fix_nvrtc():
    # 일부 Colab 이미지(새 CUDA)에서 GPU 정수 텐서의 prod가 런타임 컴파일(NVRTC) 도중 내장 라이브러리를 못 찾아 실패함.
    # 1) 그 라이브러리를 경로로 직접 불러오고, 2) 그래도 안 되면 정수 prod만 CPU에서 계산하도록 우회 (결과는 완전히 같음)
    if not torch.cuda.is_available(): return
    import glob, ctypes
    def _ok():
        try: torch.tensor([[1, 2, 3]], device="cuda").prod(-1).item(); return True, ""
        except RuntimeError as e: return False, str(e)
    good, msg = _ok()
    if good: return
    need = re.findall(r"libnvrtc-builtins\.so[\d.]*", msg)
    pats = ["/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*", "/usr/local/lib/python3*/site-packages/nvidia/**/libnvrtc-builtins.so*",
            "/usr/local/cuda*/lib64/libnvrtc-builtins.so*", "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*"]
    found = sorted({f for p in pats for f in glob.glob(p, recursive=True)})
    for f in found:
        if not need or any(os.path.basename(f).startswith(n) for n in need):
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
    if _ok()[0]: print("✅ NVRTC 내장 라이브러리를 직접 불러와 해결:", need); return
    _orig_prod = torch.Tensor.prod
    def _prod(self, *a, **k):
        if self.is_cuda and not (self.is_floating_point() or self.is_complex()):
            return _orig_prod(self.cpu(), *a, **k).to(self.device)
        return _orig_prod(self, *a, **k)
    torch.Tensor.prod = _prod
    print("⚠️ GPU 정수 prod를 CPU에서 계산하도록 우회 (결과 동일) | 확인:", _ok()[0], "| 찾은 라이브러리:", found or "없음")
_fix_nvrtc()

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, "dev.csv")) if os.path.exists(os.path.join(DATA_DIR, "dev.csv")) else None
for c in ["question", "a", "b", "c", "d"]:
    train_df[c] = train_df[c].astype(str); test_df[c] = test_df[c].astype(str)
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df.answer.unique()) <= set("abcd"), train_df.answer.unique()
assert list(test_df.id) == list(sub_df.id), "test.csv 와 sample_submission.csv 의 id 순서가 다릅니다"
print(f"train {len(train_df):,} | test {len(test_df):,} | dev {0 if dev_df is None else len(dev_df):,}")
print("정답 분포:", train_df.answer.value_counts().sort_index().to_dict())

def img_path(rel):            # CSV의 path 컬럼은 DATA_DIR 기준 상대경로
    return os.path.join(DATA_DIR, rel)

## 5. 고정 검증셋 분할 (Stage 1과 동일)

In [ ]:
EXPECTED_SPLIT_MD5 = "df05e4bc19d27ebceeef2570672149c2"

def make_split(df, val_size=VAL_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx)
    mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)

tr_df, val_df = make_split(train_df)
split_md5 = hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest()
print(f"train {len(tr_df):,} / valid {len(val_df):,} | split md5 = {split_md5}")
print("valid 정답 분포:", val_df.answer.value_counts().sort_index().to_dict())
if split_md5 == EXPECTED_SPLIT_MD5:
    print("✅ 팀 공통 분할과 일치")
else:
    print("⚠️⚠️ 검증 분할이 기준과 다릅니다 → 팀원 간 점수 비교 불가. train.csv 원본/SEED/VAL_SIZE를 확인하세요.")
val_df[["id"]].to_csv(os.path.join(OUT_DIR, "valid_ids.csv"), index=False)

## 6. 프롬프트 (Stage 1과 동일)

In [ ]:
PROMPT_VERSION = "baseline_v1"
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
LETTERS = "abcd"

def build_mc_prompt(question, options):
    a, b, c, d = options
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, perm=(0, 1, 2, 3)):
    # perm[k] = 화면의 k번째 자리에 놓을 '원래 선지'의 인덱스
    opts = [str(row[LETTERS[j]]) for j in perm]
    user_text = build_mc_prompt(str(row["question"]), opts)
    if globals().get("USE_SYSTEM_ROLE", True):
        return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    # system 역할이 없는 템플릿(일부 Gemma 등): 지시문을 user 메시지 앞에 붙임
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_INSTRUCT + "\n\n" + user_text}]}]

def load_image(rel):
    im = Image.open(img_path(rel))
    im = ImageOps.exif_transpose(im)      # 촬영 방향(EXIF) 보정 — 없으면 그대로
    return im.convert("RGB")

## 7. 모델 로드 + LoRA 부착
LoRA는 **언어 모델 쪽 선형층**(어텐션·MLP, Qwen3.5의 선형 어텐션 투영 포함)에만 붙이고 비전 타워는 얼려 둡니다.

In [ ]:
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel, set_peft_model_state_dict, load_peft_weights, prepare_model_for_kbit_training
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

import gc
for _v in ("model", "base", "optimizer", "scheduler", "params", "loss", "enc", "logits", "out", "gnorm"):   # 이전 모델·중단된 학습의 그래프가 GPU에 남아 OOM 나는 것 방지
    if _v in globals(): del globals()[_v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)
    BF16_NATIVE = cap[0] >= 8
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
else:
    GPU_NAME, BF16_NATIVE, VRAM_GB = "cpu", False, 0.0
if DEVICE == "cpu":   DTYPE, use_4bit = torch.float32, False
elif BF16_NATIVE:
    _big = re.search(r"(?<![\d.])(2\d|3\d|7\d)B", str(MODEL_ID)) is not None
    DTYPE, use_4bit = torch.bfloat16, (USE_4BIT is True or (USE_4BIT == "auto" and (VRAM_GB < 30 or _big)))
else:                 DTYPE, use_4bit = torch.float16, (USE_4BIT is not False)
USE_FP16_SCALER = (DEVICE == "cuda" and not BF16_NATIVE)
BS = (8 if VRAM_GB >= 38 else 4 if VRAM_GB >= 20 else 2) if BATCH_SIZE == "auto" else int(BATCH_SIZE)
print(f"GPU: {GPU_NAME} ({VRAM_GB:.0f}GB) | dtype={DTYPE} | 4bit={use_4bit} | 평가 배치={BS} | 학습 배치={MICRO_BS}x{GRAD_ACCUM}")

import shutil
RUN_DIR = os.path.join(OUT_DIR, RUN_NAME + ("_dry" if DRY_RUN else ""))
CKPT_DIR = os.path.join(RUN_DIR, "ckpt_last")
if DRY_RUN and os.path.exists(RUN_DIR): shutil.rmtree(RUN_DIR)          # 점검용 실행은 항상 처음부터
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(CKPT_DIR) and os.path.exists(os.path.join(CKPT_DIR + "_tmp", "trainer_state.pt")):
    os.replace(CKPT_DIR + "_tmp", CKPT_DIR)                              # 저장 직후 교체 단계에서 끊긴 경우 복구

load_kw = dict(dtype=DTYPE, attn_implementation="sdpa")
if DEVICE == "cuda": load_kw["device_map"] = {"": 0}
if use_4bit:
    load_kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE,
        llm_int8_skip_modules=["lm_head", ".*visual.*", ".*vision_tower.*", ".*vision_model.*", ".*multi_modal_projector.*", ".*embed_vision.*", ".*audio_tower.*", ".*embed_audio.*"])

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
tok = processor.tokenizer
FAMILY = "gemma4" if type(processor).__name__.startswith("Gemma4") else "qwen"     # 프로세서 계열 (해상도 인자·이미지 전달 방식이 다름)
IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
if IMAGE_TOKEN_ID is None: IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(getattr(processor, "image_token", "<|image_pad|>"))
_tmpl = processor.chat_template if isinstance(processor.chat_template, str) else json.dumps(processor.chat_template)
TEMPLATE_KW = {"enable_thinking": False} if "enable_thinking" in (_tmpl or "") else {}
try:
    processor.apply_chat_template(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW); USE_SYSTEM_ROLE = True
except Exception as _e:
    USE_SYSTEM_ROLE = False; print("system 역할 미지원 템플릿 → 지시문을 user 메시지 앞에 붙입니다")
print(f"계열: {FAMILY} | 이미지 토큰 id: {IMAGE_TOKEN_ID} | chat template 추가 인자: {TEMPLATE_KW} | system 역할: {USE_SYSTEM_ROLE}")
t0 = time.time()
base = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kw)
base.config.use_cache = False
if use_4bit and not BF16_NATIVE:
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
else:
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
print(f"모델 로드 {time.time() - t0:.0f}s | {type(base).__name__}")

LORA_TARGETS = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj)"
RESUMING = os.path.exists(os.path.join(CKPT_DIR, "trainer_state.pt"))
if RESUMING:
    model = PeftModel.from_pretrained(base, os.path.join(CKPT_DIR, "adapter"), is_trainable=True)
    print("↩️ 체크포인트에서 어댑터를 불러왔습니다:", CKPT_DIR)
else:
    torch.manual_seed(TRAIN_SEED)                                        # LoRA 초기화 시드
    model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                                            target_modules=LORA_TARGETS))
# PEFT는 체크포인팅이 켜져 있으면 '입력 임베딩 출력에 grad 요구' 훅을 거는데, transformers 5.x에서는 이 훅이 비전 타워에도 걸려
# 얼려 둔 비전 타워 전체가 역전파 그래프에 들어갑니다(메모리·시간 낭비). 비재진입(non-reentrant) 체크포인팅에서는 필요 없으므로 제거합니다.
if hasattr(base, "disable_input_require_grads") and getattr(base, "_require_grads_hooks", None):
    base.disable_input_require_grads(); print("입력-grad 훅 제거 (비전 타워는 역전파에서 제외)")
if "qwen3_5" in str(getattr(base.config, "model_type", "")):
    import importlib.util
    if importlib.util.find_spec("fla") is None:
        print("⚠️⚠️ flash-linear-attention이 없습니다 → 선형 어텐션이 순수 PyTorch로 돌아 OOM/저속 위험. 2번 셀을 실행하고 '세션 다시 시작' 후 처음부터 실행하세요.")
    else:
        print("✅ flash-linear-attention 커널 사용")
model.print_trainable_parameters()
n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
n_lora_vis = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A") and any(k in n for k in ("visual", "vision_tower", "vision_model")))
print(f"LoRA가 붙은 층: {n_lora}개 (비전 타워 {n_lora_vis}개 — 0이어야 정상)")
assert n_lora > 0 and n_lora_vis == 0

LETTER_IDS = []                                     # 첫 번째(소문자·공백 없음)가 학습 목표, 나머지 변형은 확률 집계용
for L in LETTERS:
    ids = []
    for s in (L, L.upper(), " " + L, " " + L.upper()):
        t = tok.encode(s, add_special_tokens=False)
        if len(t) == 1 and t[0] not in ids: ids.append(t[0])
    assert ids and len(tok.encode(L, add_special_tokens=False)) == 1, f"'{L}' 가 단일 토큰이 아닙니다"
    LETTER_IDS.append(ids)
print("글자 토큰 id:", dict(zip(LETTERS, LETTER_IDS)))

## 8. 채점 함수 (Stage 1과 동일한 로짓 비교) + 순열 TTA
`TTA_PERMS`는 선지를 한 칸씩 밀어 보여주는 4가지 순서입니다. 네 번 채점해 확률을 평균하면 "자리 때문에 생긴 흔들림"이 상쇄됩니다.

In [ ]:
IDENTITY = (0, 1, 2, 3)
TTA_PERMS = [(0, 1, 2, 3), (1, 2, 3, 0), (2, 3, 0, 1), (3, 0, 1, 2)]

def set_resolution(preset):
    global RES_PRESET, PIXEL_KW
    RES_PRESET = preset
    if FAMILY == "gemma4":
        PIXEL_KW = {"max_soft_tokens": int(RES_PRESETS_GEMMA4[preset])}
    else:
        mn, mx = RES_PRESETS[preset]
        PIXEL_KW = {"min_pixels": int(mn), "max_pixels": int(mx)}
set_resolution(TRAIN_PRESET)

def render(row, perm=IDENTITY):
    return processor.apply_chat_template(build_messages(row, perm), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)

from PIL import ImageEnhance
_AUG_ON = False                                   # train_micro_step 안에서만 True
AUG_RNG = np.random.RandomState(TRAIN_SEED + 12345)
def augment_image(im, rng=None):
    # 글자를 읽는 과제라 기하 변형(반전·회전·자르기)은 쓰지 않고, 촬영 조건에 해당하는 밝기·대비·채도·감마·JPEG 화질만 흔듦
    rng = rng or AUG_RNG
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.8, 1.2))
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.8, 1.2))
    im = ImageEnhance.Color(im).enhance(rng.uniform(0.9, 1.1))
    g = rng.uniform(0.85, 1.18); im = im.point(lambda v, g=g: int(255 * (v / 255) ** g))
    if rng.rand() < 0.5:
        import io; buf = io.BytesIO(); im.save(buf, "JPEG", quality=int(rng.uniform(60, 95))); buf.seek(0); im = Image.open(buf).convert("RGB")
    return im

def encode_batch(rows, perms):
    texts  = [render(r, p) for r, p in zip(rows, perms)]
    images = [load_image(r["path"]) for r in rows]
    if _AUG_ON and IMG_AUG: images = [augment_image(im) for im in images]
    if FAMILY == "gemma4": images = [[im] for im in images]          # Gemma 프로세서는 샘플별 이미지 리스트를 받음
    return processor(text=texts, images=images, padding=True, return_tensors="pt", **PIXEL_KW)

def _last_logits(enc):
    enc = enc.to(model.device)
    with torch.autocast(device_type="cuda", dtype=DTYPE, enabled=(DEVICE == "cuda")):
        try:    out = model(**enc, logits_to_keep=1, use_cache=False)
        except TypeError: out = model(**enc)
    return out.logits[:, -1, :].float()

def unpermute(prob_slot, perms):
    probs = np.zeros_like(prob_slot)
    for i, p in enumerate(perms):
        for k, j in enumerate(p): probs[i, j] = prob_slot[i, k]
    return probs

@torch.inference_mode()
def score_rows(rows, perms=None):
    global BS
    perms = perms or [IDENTITY] * len(rows)
    enc = logits = None; oom = False
    try:
        enc = encode_batch(rows, perms)
        n_img_tok = (enc["input_ids"] == IMAGE_TOKEN_ID).sum(dim=1).tolist()
        logits = _last_logits(enc)
        if not torch.isfinite(logits).all(): raise FloatingPointError("로짓에 NaN/Inf 발생")
    except torch.cuda.OutOfMemoryError:
        oom = True
    if oom:
        del enc, logits; torch.cuda.empty_cache()
        if len(rows) == 1: raise RuntimeError("배치 1에서도 GPU 메모리 부족")
        h = len(rows) // 2; BS = max(1, min(BS, h)); print(f"  ⚠️ OOM → 평가 배치를 {BS}(으)로 줄입니다")
        p1, m1, t1 = score_rows(rows[:h], perms[:h]); p2, m2, t2 = score_rows(rows[h:], perms[h:])
        return np.concatenate([p1, p2]), np.concatenate([m1, m2]), t1 + t2
    logp = torch.log_softmax(logits, dim=-1)
    slot = torch.stack([torch.logsumexp(logp[:, ids], dim=1) for ids in LETTER_IDS], dim=1)
    mass = slot.exp().sum(dim=1).cpu().numpy()
    return unpermute(torch.softmax(slot, dim=1).cpu().numpy().astype(np.float64), perms), mass, n_img_tok

def score_rows_tta(rows, tta):
    ps, ms, nt = [], [], None
    for perm in (TTA_PERMS if tta else [IDENTITY]):
        p, m, nt = score_rows(rows, [perm] * len(rows)); ps.append(p); ms.append(m)
    return np.mean(ps, axis=0), np.mean(ms, axis=0), nt

def predict_df(df, tag, tta=False, resume=True, verbose=True, save_every=25):
    # df 전체를 채점. resume=True면 CSV에 이어 쓰며, 같은 tag로 다시 부르면 남은 문항만 처리
    was_training = model.training; model.eval()
    out_path = os.path.join(RUN_DIR, f"preds_{tag}__{RES_PRESET}{'__tta4' if tta else ''}__{len(df)}.csv")
    done_ids = set(pd.read_csv(out_path).id) if (resume and os.path.exists(out_path)) else set()
    if not resume and os.path.exists(out_path): os.remove(out_path)
    todo = df[~df.id.isin(done_ids)]
    if SKIP_MISSING_IMAGES: todo = todo[todo.path.map(lambda p: os.path.exists(img_path(p)))]
    rows = todo.to_dict("records"); recs, t0, n_done, nb = [], time.time(), 0, 0
    if verbose: print(f"[{os.path.basename(out_path)}] 전체 {len(df):,} | 완료 {len(done_ids):,} | 남음 {len(rows):,}")
    while n_done < len(rows):
        batch = rows[n_done: n_done + BS]
        probs, mass, ntok = score_rows_tta(batch, tta)
        for r, p, m, t in zip(batch, probs, mass, ntok):
            recs.append({"id": r["id"], "pred": LETTERS[int(p.argmax())], "p_a": p[0], "p_b": p[1], "p_c": p[2], "p_d": p[3],
                         "letter_mass": float(m), "img_tokens": int(t)})
        n_done += len(batch); nb += 1
        if nb % save_every == 0 or n_done >= len(rows):
            pd.DataFrame(recs).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False); recs = []
            if verbose:
                el = time.time() - t0
                print(f"  {n_done:>5}/{len(rows)} | {el / n_done:.2f}s/문항 | 남은 시간 ~{(len(rows) - n_done) * el / n_done / 60:.0f}분", flush=True)
    if was_training: model.train()
    return pd.read_csv(out_path).drop_duplicates("id", keep="last"), out_path

CATS = {"전화번호": r"전화|번호|연락처", "가격/금액": r"가격|얼마|금액|요금|할인", "이름/상호/제목": r"이름|상호|제목|브랜드|가게|매장|간판|업체",
        "날짜/시간": r"날짜|언제|시간|기간|요일|몇 시|년도|연도", "위치/방향/층": r"어디|위치|방향|몇 층|층에|왼쪽|오른쪽|위쪽|아래",
        "개수/숫자": r"몇 개|몇 명|몇 번|개수|몇 가지|숫자", "색상": r"색|컬러", "부정형": r"아닌|않는|없는|틀린|잘못"}

def evaluate(df, tag, tta=False, resume=True, verbose=True, by_type=True):
    pred, _ = predict_df(df, tag, tta=tta, resume=resume, verbose=verbose)
    m = df.merge(pred, on="id", how="inner"); m["correct"] = (m.pred == m.answer)
    acc = float(m.correct.mean()); n = len(m); se = math.sqrt(max(acc * (1 - acc), 1e-9) / n)
    print(f"▶ {tag} | {RES_PRESET}{' +TTA4' if tta else ''} | 정확도 {acc * 100:.2f}% (n={n}, ±{1.96 * se * 100:.1f}%p) | letter_mass {m.letter_mass.mean():.3f}")
    if by_type:
        for k, pat in CATS.items():
            sel = m.question.str.contains(pat)
            if sel.sum() >= 10: print(f"    {k:<12} {m.correct[sel].mean() * 100:5.1f}%  (n={int(sel.sum())})")
    lp = os.path.join(OUT_DIR, "results_log_s2.csv")
    log = {"time": datetime.datetime.now().strftime("%m-%d %H:%M"), "run": f"{RUN_NAME}{'_dry' if DRY_RUN else ''}/{tag}", "model": str(MODEL_ID),
           "res": RES_PRESET, "tta": int(tta), "n": n, "acc": round(acc * 100, 2), "ci95": round(1.96 * se * 100, 2),
           "letter_mass": round(float(m.letter_mass.mean()), 3), "gpu": GPU_NAME, "split_md5": split_md5[:8]}
    pd.DataFrame([log]).to_csv(lp, mode="a", header=not os.path.exists(lp), index=False)
    return acc, m

## 9. 학습 준비: 배치 구성 · 손실 · 체크포인트
- `epoch_plan`: epoch마다 문항 순서와 선지 순열을 **시드로 결정** → 중간에 끊겼다 이어도 똑같은 데이터 순서가 재현됩니다.
- 손실: 추론과 같은 프롬프트의 **마지막 위치 로짓**에, 섞인 순서 기준의 정답 글자(소문자 토큰)로 교차 엔트로피.

In [ ]:
train_src = train_df if TRAIN_ON_FULL else tr_df
if SKIP_MISSING_IMAGES: train_src = train_src[train_src.path.map(lambda p: os.path.exists(img_path(p)))]
if DRY_RUN:       train_src = train_src.head(3 * MICRO_BS * GRAD_ACCUM)
elif TRAIN_LIMIT: train_src = train_src.head(int(TRAIN_LIMIT))
if EXTRA_TRAIN_CSV:                                   # 추가 학습 데이터(예: dev 의사 라벨)는 위 제한과 별개로 뒤에 붙임
    _need = ["id", "path", "question", "a", "b", "c", "d", "answer"]
    extra = pd.read_csv(EXTRA_TRAIN_CSV); assert all(c in extra.columns for c in _need), f"추가 데이터 열이 부족합니다: {list(extra.columns)}"
    for c in ["question", "a", "b", "c", "d"]: extra[c] = extra[c].astype(str)
    extra["answer"] = extra["answer"].astype(str).str.strip().str.lower(); assert set(extra.answer.unique()) <= set("abcd"), extra.answer.unique()
    assert not (set(extra.id) & set(val_df.id)), "추가 데이터에 검증 문항이 섞여 있습니다"
    assert extra.path.map(lambda p: os.path.exists(img_path(p))).all(), "추가 데이터의 이미지 경로가 없습니다"
    if DRY_RUN: extra = extra.head(2)
    train_src = pd.concat([train_src[_need], extra[_need]], ignore_index=True)
    print(f"추가 학습 데이터 {len(extra):,}문항 합침 (정답 분포 {extra.answer.value_counts().sort_index().to_dict()}) → 학습 총 {len(train_src):,}문항")
train_rows = train_src.to_dict("records")
STEP_SAMPLES = MICRO_BS * GRAD_ACCUM
steps_per_epoch = len(train_rows) // STEP_SAMPLES
n_epochs = 1 if DRY_RUN else EPOCHS
total_steps = steps_per_epoch * n_epochs
if MAX_STEPS and not DRY_RUN: total_steps = min(total_steps, int(MAX_STEPS))
assert total_steps > 0, "학습 샘플이 유효 배치보다 적습니다"
print(f"학습 문항 {len(train_rows):,} | 유효 배치 {STEP_SAMPLES} | epoch당 {steps_per_epoch} 스텝 | 총 {total_steps} 스텝" + (" | ⚠️ 검증셋 포함 학습(TRAIN_ON_FULL)" if TRAIN_ON_FULL else ""))

def epoch_plan(epoch):
    rng = np.random.RandomState(TRAIN_SEED * 1000 + epoch)
    order = rng.permutation(len(train_rows))
    perms = [tuple(rng.permutation(4).tolist()) if SHUFFLE_OPTIONS else IDENTITY for _ in range(len(train_rows))]
    return order, perms

def gold_slot(row, perm):                 # 섞인 화면에서 정답이 놓인 자리 (0~3)
    return perm.index(LETTERS.index(row["answer"]))

def train_micro_step(rows, perms):
    global _AUG_ON
    _AUG_ON = True
    try: enc = encode_batch(rows, perms)
    finally: _AUG_ON = False
    target = torch.tensor([LETTER_IDS[gold_slot(r, p)][0] for r, p in zip(rows, perms)], device=model.device)
    logits = _last_logits(enc)
    loss = F.cross_entropy(logits, target)
    with torch.no_grad():
        slot = torch.stack([logits[:, ids[0]] for ids in LETTER_IDS], dim=1)
        hit = (slot.argmax(1).cpu() == torch.tensor([gold_slot(r, p) for r, p in zip(rows, perms)])).float().mean().item()
    return loss, hit

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, int(total_steps * WARMUP_RATIO)), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16_SCALER)
state = {"global_step": 0, "history": [], "epoch_acc": {}, "config": {k: globals()[k] for k in
         ["MODEL_ID", "TRAIN_PRESET", "EPOCHS", "MAX_STEPS", "LR", "STEP_SAMPLES", "LORA_R", "LORA_ALPHA", "SHUFFLE_OPTIONS", "TRAIN_ON_FULL", "SEED", "TRAIN_SEED", "EXTRA_TRAIN_CSV", "IMG_AUG", "TRAIN_PRESETS"]}}

def save_checkpoint():
    tmp = CKPT_DIR + "_tmp"; os.makedirs(tmp, exist_ok=True)
    model.save_pretrained(os.path.join(tmp, "adapter"))
    rng = {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
           "cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}
    torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "state": state, "rng": rng}, os.path.join(tmp, "trainer_state.pt"))
    if os.path.exists(CKPT_DIR): shutil.rmtree(CKPT_DIR)
    os.replace(tmp, CKPT_DIR)                     # 저장 도중 끊겨도 이전 체크포인트가 깨지지 않도록 교체 방식으로

if RESUMING:
    ck = torch.load(os.path.join(CKPT_DIR, "trainer_state.pt"), map_location="cpu", weights_only=False)
    _defaults = {"TRAIN_SEED": 42, "MAX_STEPS": None, "EXTRA_TRAIN_CSV": None, "IMG_AUG": False, "TRAIN_PRESETS": None}
    same = all(ck["state"]["config"].get(k, _defaults.get(k)) == v for k, v in state["config"].items())
    assert same, f"체크포인트의 설정이 현재와 다릅니다. RUN_NAME을 바꾸거나 설정을 되돌리세요.\n저장됨: {ck['state']['config']}\n현재: {state['config']}"
    optimizer.load_state_dict(ck["optimizer"]); scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    state = ck["state"]
    random.setstate(ck["rng"]["py"]); np.random.set_state(ck["rng"]["np"]); torch.set_rng_state(ck["rng"]["torch"])
    if DEVICE == "cuda" and ck["rng"]["cuda"] is not None: torch.cuda.set_rng_state_all(ck["rng"]["cuda"])
    print(f"↩️ {state['global_step']}/{total_steps} 스텝부터 이어서 학습합니다")

## 10. 학습 루프
로그의 `loss`는 정답 글자에 대한 교차 엔트로피(처음엔 0.2 안팎에서 시작해 내려가야 정상), `train_acc`는 학습 배치에서의 정답률입니다.

In [ ]:
_STOP_AT = None      # (테스트용) 이 스텝에서 강제 중단
quick_df = val_df.head(QUICK_EVAL_N if not DRY_RUN else 3 * BS)
if SKIP_MISSING_IMAGES: quick_df = quick_df[quick_df.path.map(lambda p: os.path.exists(img_path(p)))]
if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
model.train(); optimizer.zero_grad(set_to_none=True)
t_start, run_loss, run_hit, n_micro = time.time(), 0.0, 0.0, 0
step0 = state["global_step"]; plan_epoch = -1

for step in range(step0, total_steps):
    epoch, k = divmod(step, steps_per_epoch)
    if epoch != plan_epoch: order, perms_all = epoch_plan(epoch); plan_epoch = epoch
    idx = order[k * STEP_SAMPLES: (k + 1) * STEP_SAMPLES]
    while True:                                        # OOM이면 마이크로 배치를 절반으로 줄여 같은 스텝을 다시 시도 (유효 배치는 그대로)
        oom = False; s_loss, s_hit, s_n = 0.0, 0.0, 0
        try:
            for s in range(0, STEP_SAMPLES, MICRO_BS):
                mb = idx[s: s + MICRO_BS]
                if TRAIN_PRESETS: set_resolution(TRAIN_PRESETS[np.random.RandomState(TRAIN_SEED * 100003 + step * 97 + s).randint(len(TRAIN_PRESETS))])   # 결정적 → 재개해도 같음
                loss, hit = train_micro_step([train_rows[i] for i in mb], [perms_all[i] for i in mb])
                scaler.scale(loss / GRAD_ACCUM).backward()
                s_loss += loss.item(); s_hit += hit; s_n += 1
        except torch.cuda.OutOfMemoryError:
            oom = True
        if not oom: break
        loss = None; optimizer.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
        if MICRO_BS == 1: raise RuntimeError("마이크로 배치 1에서도 GPU 메모리 부족 → TRAIN_PRESET을 낮추거나 4bit를 켜세요.")
        MICRO_BS //= 2; GRAD_ACCUM *= 2
        print(f"  ⚠️ OOM → MICRO_BS={MICRO_BS}, GRAD_ACCUM={GRAD_ACCUM} 로 바꿔 이 스텝을 다시 실행합니다", flush=True)
    run_loss += s_loss; run_hit += s_hit; n_micro += s_n
    if TRAIN_PRESETS: set_resolution(TRAIN_PRESET)
    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
    state["global_step"] = step + 1

    if (step + 1) % LOG_EVERY == 0 or step + 1 == total_steps or DRY_RUN:
        el = time.time() - t_start; done = step + 1 - step0
        print(f"step {step + 1:>4}/{total_steps} | ep {epoch + 1} | loss {run_loss / n_micro:.4f} | train_acc {run_hit / n_micro * 100:5.1f}% | "
              f"lr {scheduler.get_last_lr()[0]:.2e} | gnorm {float(gnorm):.2f} | {el / done:.1f}s/step | 남은 시간 ~{(total_steps - step - 1) * el / done / 60:.0f}분", flush=True)
        state["history"].append({"step": step + 1, "loss": run_loss / n_micro, "train_acc": run_hit / n_micro})
        run_loss, run_hit, n_micro = 0.0, 0.0, 0
    end_of_epoch = (step + 1) % steps_per_epoch == 0 or step + 1 == total_steps      # MAX_STEPS로 중간에 끝나는 경우도 'epoch 끝'으로 취급
    if ((step + 1) % EVAL_EVERY == 0 and not end_of_epoch) or (DRY_RUN and step + 1 == total_steps):
        acc, _ = evaluate(quick_df, f"quick_step{step + 1}", resume=False, verbose=False, by_type=False)
        state["history"].append({"step": step + 1, "quick_val_acc": acc})
    if end_of_epoch and not DRY_RUN:
        acc, _ = evaluate(val_df, f"ep{epoch + 1}", resume=False, verbose=False)
        state["epoch_acc"][epoch + 1] = acc
        model.save_pretrained(os.path.join(RUN_DIR, f"adapter_ep{epoch + 1}"))
    if (step + 1) % SAVE_EVERY == 0 or end_of_epoch or step + 1 == total_steps:
        save_checkpoint()
    if _STOP_AT and step + 1 == _STOP_AT: print("강제 중단(테스트)"); break

if DEVICE == "cuda": print(f"최대 VRAM 사용: {torch.cuda.max_memory_allocated() / 2**30:.1f}GB / {VRAM_GB:.0f}GB")
if DRY_RUN and total_steps > step0:
    per_step = (time.time() - t_start) / (total_steps - step0)
    full_steps = (len(train_df if TRAIN_ON_FULL else tr_df) // STEP_SAMPLES) * EPOCHS
    print(f"\n✅ DRY RUN 통과. 본 학습 예상: {full_steps} 스텝 x {per_step:.1f}s ≈ {full_steps * per_step / 3600:.1f}시간 (+평가 약 15분). DRY_RUN=False로 바꿔 1번 셀부터 다시 실행하세요.")
json.dump({"epoch_acc": state["epoch_acc"], "history": state["history"], "config": state["config"]},
          open(os.path.join(RUN_DIR, "train_log.json"), "w"), indent=1, default=str)

## 11. 최종 평가: 가장 좋은 epoch의 어댑터 → 해상도별 · TTA 유무별 검증 점수 + 오답 리포트
`error_report_*.html`은 틀린 문항을 이미지와 함께 모은 파일입니다. Drive에서 내려받아 브라우저로 열면 되고(런타임 불필요).

In [ ]:
import base64, io, html as _html
def export_error_report(m, path, max_items=150):
    wrong = m[~m.correct].head(max_items); parts = [f"<h2>{_html.escape(os.path.basename(path))} — 오답 {int((~m.correct).sum())}건 / {len(m)}문항</h2>"]
    for _, r in wrong.iterrows():
        im = load_image(r.path); im.thumbnail((560, 560)); buf = io.BytesIO(); im.save(buf, "JPEG", quality=72)
        opts = "".join(f"<li>({k}) {_html.escape(str(r[k]))} — p={r['p_' + k]:.2f}{' <b>← 정답</b>' if k == r.answer else ''}{' <b style=color:#c00>← 예측</b>' if k == r.pred else ''}</li>" for k in LETTERS)
        parts.append(f"<div style='display:flex;gap:16px;margin:18px 0;border-top:1px solid #ccc;padding-top:12px'><img src='data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}'>"
                     f"<div><b>{r.id}</b><p>{_html.escape(str(r.question))}</p><ul>{opts}</ul></div></div>")
    open(path, "w", encoding="utf-8").write("<meta charset='utf-8'><body style='font-family:sans-serif;max-width:1100px;margin:auto'>" + "".join(parts))
    print("오답 리포트 저장:", path)

if not DRY_RUN and state["global_step"] < total_steps:
    raise RuntimeError(f"학습이 아직 끝나지 않았습니다 ({state['global_step']}/{total_steps} 스텝). 1번 셀부터 다시 실행하면 체크포인트에서 이어서 학습합니다.")
if not DRY_RUN and state["epoch_acc"]:
    best_ep = max(state["epoch_acc"], key=lambda e: (state["epoch_acc"][e], e))     # 동률이면 더 오래 학습한 쪽
    print("epoch별 검증 정확도:", {e: round(a * 100, 2) for e, a in state["epoch_acc"].items()}, "→ 사용할 어댑터: epoch", best_ep)
    set_peft_model_state_dict(model, load_peft_weights(os.path.join(RUN_DIR, f"adapter_ep{best_ep}"), device="cpu"))
    FINAL_TAG = f"final_ep{best_ep}"
else:
    FINAL_TAG = "final_dry"
model.eval()
final_eval_df = val_df if not DRY_RUN else quick_df
summary = []
for preset in (EVAL_PRESETS if not DRY_RUN else [TRAIN_PRESET]):
    set_resolution(preset)
    if not DRY_RUN and preset == TRAIN_PRESET:                    # epoch 끝 평가와 같은 모델·해상도 → 그 결과를 그대로 재사용
        _src = os.path.join(RUN_DIR, f"preds_ep{best_ep}__{preset}__{len(final_eval_df)}.csv")
        _dst = os.path.join(RUN_DIR, f"preds_{FINAL_TAG}__{preset}__{len(final_eval_df)}.csv")
        if os.path.exists(_src) and not os.path.exists(_dst): shutil.copy(_src, _dst)
    acc, m = evaluate(final_eval_df, FINAL_TAG, tta=False)
    summary.append({"해상도": preset, "TTA": False, "정확도(%)": round(acc * 100, 2)})
    export_error_report(m, os.path.join(RUN_DIR, f"error_report_{FINAL_TAG}_{preset}.html"))
BEST_PRESET = max(summary, key=lambda s: s["정확도(%)"])["해상도"]        # 동률이면 앞에 적은(=더 싼) 해상도
if EVAL_TTA:
    set_resolution(BEST_PRESET)
    acc, _ = evaluate(final_eval_df, FINAL_TAG, tta=True, by_type=False)
    summary.append({"해상도": BEST_PRESET, "TTA": True, "정확도(%)": round(acc * 100, 2)})
print("\n" + pd.DataFrame(summary).to_string(index=False))
print(f"→ 제출 권장 해상도: {BEST_PRESET}")
if TRAIN_ON_FULL: print("⚠️ TRAIN_ON_FULL=True: 검증셋을 학습에 썼으므로 위 점수는 의미가 없습니다 (제출용 모델).")
set_resolution(TRAIN_PRESET)

## 12. test 추론 → 제출 파일 (이어하기 지원)

In [ ]:
if MAKE_SUBMISSION and not DRY_RUN:
    test_preset = BEST_PRESET if TEST_PRESET == "auto" else TEST_PRESET
    set_resolution(test_preset)
    pred, _ = predict_df(test_df, f"{FINAL_TAG}_test", tta=TEST_TTA)
    sub = sub_df[["id"]].merge(pred[["id", "pred"]], on="id", how="left").rename(columns={"pred": "answer"})
    assert len(sub) == len(sub_df) and list(sub.id) == list(sub_df.id), "id 순서/개수 불일치"
    assert sub.answer.notna().all(), f"예측 누락 {int(sub.answer.isna().sum())}건"
    assert set(sub.answer.unique()) <= set(LETTERS), sub.answer.unique()
    sp = os.path.join(RUN_DIR, f"submission_{RUN_NAME}_{FINAL_TAG}_{test_preset}{'_tta4' if TEST_TTA else ''}.csv"); sub.to_csv(sp, index=False)
    print("저장:", sp, "| 예측 분포:", sub.answer.value_counts().sort_index().to_dict())
    set_resolution(TRAIN_PRESET)

## 13. 결과 요약

In [ ]:
print("RUN:", RUN_NAME, "| epoch별 검증 정확도:", {e: round(a * 100, 2) for e, a in state["epoch_acc"].items()})
h = pd.DataFrame([x for x in state["history"] if "loss" in x])
if len(h): print("loss 추이(처음→끝):", [round(v, 4) for v in h.loss.iloc[:: max(1, len(h) // 8)].tolist()])
q = [(x["step"], round(x["quick_val_acc"] * 100, 1)) for x in state["history"] if "quick_val_acc" in x]
if q: print("학습 중 빠른 검증(step, 정확도%):", q)
lp = os.path.join(OUT_DIR, "results_log_s2.csv")
if os.path.exists(lp):
    L = pd.read_csv(lp); L = L[L.run.astype(str).str.startswith(RUN_NAME)]
    print(L[[c for c in ["time", "run", "res", "tta", "n", "acc", "ci95", "letter_mass"] if c in L.columns]].tail(12).to_string(index=False))